In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import pickle
from catboost import CatBoostRegressor, Pool
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

model_dir = "../models"
data_dir = "../data"


def model_path(filename):
    return os.path.join(model_dir, filename)

def data_path(filename):
    return os.path.join(data_dir, filename)


In [2]:
cat_features = ["user_city", "user_type", "poi_city", "poi_type"]

TYPE_RELATIONS = {
    "Coffee Shop": ["Cafe", "Tea Shop", "Bakery"],
    "Restaurant": ["Bar/Pub", "Vegetarian", "Bakery"],
    "Bar/Pub": ["Restaurant"],
    "Natural": ["Attraction", "Entertainment"],
    "Attraction": ["Natural", "Cultural", "Historical"],
    "Cultural": ["Attraction", "Historical"],
    "Historical": ["Attraction", "Cultural"],
    "Hotel": ["Villa", "Resort", "Homestay", "Hostel", "Apartment"],
    "Homestay": ["Hotel", "Hostel"],
    "Hostel": ["Homestay", "Hotel"],
    "Villa": ["Hotel", "Resort"],
    "Resort": ["Hotel", "Villa"],
    "Apartment": ["Hotel"],
    "Shopping": ["Entertainment"],
    "Entertainment": ["Shopping", "Natural"],
    "Vegetarian": ["Restaurant"],
    "Tea Shop": ["Coffee Shop"],
}

TYPE_RELATIONS_LOWER = {k.lower(): [v.lower() for v in vals] for k, vals in TYPE_RELATIONS.items()}

def is_similar_type(user_type, poi_type):
    u = user_type.lower().strip()
    p = poi_type.lower().strip()
    if u == p:
        return True
    return u in TYPE_RELATIONS_LOWER and p in TYPE_RELATIONS_LOWER[u]

def similarity_weight(user_type, poi_type):
    u = user_type.lower().strip()
    p = poi_type.lower().strip()
    if u == p:
        return 1.20
    if is_similar_type(u, p):
        return 1.05
    return 0.90

def generate_price_rules(df, medium_cap_default=200_000):
    quantiles = df.groupby("category")["price"].quantile([0.25, 0.5, 0.75]).unstack()
    quantiles.columns = ["Q1", "Q2", "Q3"]

    rules = {}

    for cat, row in quantiles.iterrows():
        q1, q2, q3 = row["Q1"], row["Q2"], row["Q3"]

        if q1 == 0 and q2 == 0 and q3 == 0:
            rules[cat.lower()] = {
                "type": "skewed_zero",
                "L0": 0,
                "L1": medium_cap_default,
                "details": row.to_dict()
            }
        else:
            rules[cat.lower()] = {
                "type": "normal",
                "L0": q1,
                "L1": q3,
                "details": row.to_dict()
            }

    return rules


def map_price_level(category, price, rules):
    cat = category.lower()
    if cat not in rules:
        return 1

    r = rules[cat]

    if r["type"] == "skewed_zero":
        if price == 0:
            return 0
        if price <= r["L1"]:
            return 1
        return 2

    # normal category
    if price <= r["L0"]:
        return 0
    if price <= r["L1"]:
        return 1
    return 2

In [3]:
def load_and_clean_data(file_path="data/POI.csv"):
    df = pd.read_csv(file_path)
    df = df[["poi_id","name","city_norm","type","price","rating","latitude","longitude", "category"]]
    
    rules = generate_price_rules(df)
    
    df["price_level"] = df.apply(
        lambda r: map_price_level(r["category"], r["price"], rules),
        axis=1
    )
    df["city_norm"] = df["city_norm"].str.strip().str.lower()
    df["type"] = df["type"].str.strip().str.lower()
    return df


In [4]:
train_df = pd.read_csv("../data/recommender_training.csv")
print("Loaded train_df:", train_df.shape)


# ===== LINEAR REGRESSION (LIBRARY VERSION) =====

X_num = train_df[["user_price","poi_price","rating","latitude","longitude"]].values
y = train_df["label"].values

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_num, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_lr)
X_test_lr  = scaler.transform(X_test_lr)

lr = LinearRegression()
lr.fit(X_train_lr, y_train_lr)

pred_lr = lr.predict(X_test_lr)

with open("../models/linear.pkl", "wb") as f:
    pickle.dump({
        "model": lr,
        "scaler": scaler
    }, f)

print("✓ Saved Linear Regression model.")

Loaded train_df: (117936, 11)
✓ Saved Linear Regression model.


In [5]:
# ===== RANDOM FOREST =====

df_rf = train_df.copy()

for col in ["user_city","user_type","poi_city","poi_type"]:
    df_rf[col] = df_rf[col].astype("category").cat.codes

X_rf = df_rf[[
    "user_city","user_type","user_price",
    "poi_city","poi_type","poi_price",
    "rating","latitude","longitude"
]].values

y_rf = df_rf["label"].values

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train_rf)
pred_rf = rf_model.predict(X_test_rf)

with open("../models/random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

print("✓ Saved Random Forest model.")


✓ Saved Random Forest model.


In [6]:
# ===== CATBOOST =====
cat_features = ["user_city","user_type","poi_city","poi_type"]

X_cat = train_df[[
    "user_city","user_type","user_price",
    "poi_city","poi_type","poi_price",
    "rating","latitude","longitude"
]]

y_cat = train_df["label"]

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y_cat, test_size=0.2, random_state=42
)

for col in cat_features:
    X_train_cat[col] = X_train_cat[col].fillna("unknown").astype(str)
    X_test_cat[col]  = X_test_cat[col].fillna("unknown").astype(str)

cat = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    verbose=False
)

train_pool = Pool(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features
)

cat.fit(train_pool)

pred_cat = cat.predict(X_test_cat)

with open("../models/catboost.pkl", "wb") as f:
    pickle.dump(cat, f)

print("✓ Saved CatBoost model.")


✓ Saved CatBoost model.


In [7]:
def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

results = [
    evaluate("Linear Regression", y_test_lr, pred_lr),
    evaluate("Random Forest", y_test_rf, pred_rf),
    evaluate("CatBoost", y_test_cat, pred_cat)
]

pd.DataFrame(results)

,Model,RMSE,MAE
0,Linear Regression,0.111907,0.081096
1,Random Forest,0.021840,0.007636
2,CatBoost,0.003611,0.001003


In [8]:
def get_model(model_name):
    if model_name == "linear":
        # load sklearn LinearRegression + scaler
        with open("../models/linear.pkl", "rb") as f:
            obj = pickle.load(f)
        return obj   # {"model": lr, "scaler": scaler}

    elif model_name == "random_forest":
        with open("../models/random_forest.pkl", "rb") as f:
            model = pickle.load(f)
        return model

    elif model_name == "catboost":
        with open("../models/catboost.pkl", "rb") as f:
            model = pickle.load(f)
        return model

    else:
        raise ValueError("Unknown model name")

def prepare_input(df, model_name, model_obj=None):

    if model_name == "linear":
        X = df[[
            "user_price","poi_price",
            "rating","latitude","longitude"
        ]].values

        # apply scaler đã train
        scaler = model_obj["scaler"]
        return scaler.transform(X)

    elif model_name == "random_forest":
        df2 = df.copy()

        for col in ["user_city","user_type","poi_city","poi_type"]:
            df2[col] = df2[col].astype("category").cat.codes

        return df2[[
            "user_city","user_type","user_price",
            "poi_city","poi_type","poi_price",
            "rating","latitude","longitude"
        ]].values

    elif model_name == "catboost":
        return df[[
            "user_city","user_type","user_price",
            "poi_city","poi_type","poi_price",
            "rating","latitude","longitude"
        ]]

    else:
        raise ValueError("Unknown model name")

In [33]:
def get_top_poi(df_raw, city, user_type, price, model_name="catboost", top_k=10):

    model_obj = get_model(model_name)
    city = city.lower().strip()

    if isinstance(user_type, str):
        user_types = [user_type.lower().strip()]
    else:
        user_types = [ut.lower().strip() for ut in user_type]

    subset = df_raw[df_raw["city_norm"] == city]
    if subset.empty:
        return "City not found!"

    rows = []
    for _, poi in subset.iterrows():
        for ut in user_types:
            rows.append({
                "user_city": city,
                "user_type": ut,
                "user_price": price,
                "poi_city": poi["city_norm"],
                "poi_type": poi["type"],
                "poi_price": poi["price_level"],
                "rating": poi["rating"],
                "latitude": poi["latitude"],
                "longitude": poi["longitude"],
                "poi_id": poi["poi_id"],
                "name": poi["name"]
            })

    inf = pd.DataFrame(rows)

    if model_name == "linear":
        inf[[
            "user_price","poi_price","rating","latitude","longitude"
        ]] = inf[[
            "user_price","poi_price","rating","latitude","longitude"
        ]].fillna(0)

    X = prepare_input(inf, model_name, model_obj)

    if model_name == "linear":
        inf["score"] = model_obj["model"].predict(X)
    else:
        inf["score"] = model_obj.predict(X)

    agg = (
        inf.groupby(["poi_id", "name"], as_index=False)["score"]
           .mean()
           .sort_values("score", ascending=False)
           .head(top_k)
    )
    agg=agg.merge(df_raw[["poi_id", "city_norm","type","price_level"]],on="poi_id",how="left")

    return agg


In [38]:
df_raw = load_and_clean_data("../data/POI.csv")

for m in ["linear", "random_forest", "catboost"]:
    print("\n=== MODEL:", m.upper(), "===")
    print(get_top_poi(df_raw, "ha noi", ["bar/pub"], 1, model_name=m, top_k=5))


=== MODEL: LINEAR ===
            poi_id                                               name  \
0       hotel01847                               Minasi Premium Hotel   
1  restaurant01169                                 Cái Mâm Restaurant   
2  attraction00632                                   La Belle Vie Spa   
3       hotel01872                    La Siesta Premium Hang Be Hotel   
4  attraction00560  Xe 2 tầng Hà Nội - Hop on hop off bus Vietnam ...   

      score city_norm        type  price_level  
0  0.710100    ha noi       hotel            2  
1  0.710099    ha noi  restaurant            2  
2  0.710099    ha noi  attraction            2  
3  0.710099    ha noi       hotel            2  
4  0.710098    ha noi  attraction            2  

=== MODEL: RANDOM_FOREST ===
            poi_id                        name     score city_norm  \
0       hotel01865        TA Hotel & Apartment  0.986960    ha noi   
1       hotel01869     iStay Hotel Apartment 5  0.981660    ha noi   
2   